# Total Energy K-Grid Convergence

Run a total-energy convergence workflow for a material and return the converged k-grid.

<h2 style="color:green">Usage</h2>

1. Set the material, convergence, and compute parameters in cell 1.2.
2. Click `Run` > `Run All`.
3. Wait for the job to finish.
4. Read the converged k-grid from the final cells.

## Summary

1. Set up environment and parameters.
2. Authenticate and initialize the API client.
3. Load and save the material.
4. Create a total-energy workflow and add convergence directly in the notebook.
5. Create and submit the job.
6. Read `job.scopeTrack` from the finished job.
7. Extract the convergence series and final converged k-grid.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install("mat3ra-api-examples", deps=False)
    await micropip.install("mat3ra-utils")
    from mat3ra.utils.jupyterlite.packages import install_packages

    await install_packages("api_examples")


### 1.2. Set parameters and configurations for the convergence job


In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 1. Organization / account selection
ORGANIZATION_NAME = None

# 2. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"

# 3. Workflow parameters
APPLICATION_NAME = "espresso"
WORKFLOW_SEARCH_TERM = "total_energy.json"
MY_WORKFLOW_NAME = "Total Energy Convergence"

# 4. Convergence parameters
KGRID_CONVERGENCE_TYPE = "uniform"  # "uniform" or "non-uniform"
UNIFORM_KGRID_INITIAL = 1
UNIFORM_KGRID_INCREMENT = 1
NON_UNIFORM_KGRID_INITIAL = [2, 2, 1]
NON_UNIFORM_KGRID_INCREMENT = 1
RESULT_INITIAL = 0
TOLERANCE = 1e-3
MAX_OCCURRENCES = 10
CONVERGENCE_CONDITION = "abs((prev_result-total_energy)/total_energy)"

if KGRID_CONVERGENCE_TYPE == "uniform":
    CONVERGENCE_PARAMETER = "N_k"
    PARAMETER_INITIAL = UNIFORM_KGRID_INITIAL
    PARAMETER_INCREMENT = UNIFORM_KGRID_INCREMENT
elif KGRID_CONVERGENCE_TYPE == "non-uniform":
    CONVERGENCE_PARAMETER = "N_k_nonuniform"
    PARAMETER_INITIAL = NON_UNIFORM_KGRID_INITIAL
    PARAMETER_INCREMENT = NON_UNIFORM_KGRID_INCREMENT
else:
    raise ValueError("KGRID_CONVERGENCE_TYPE must be 'uniform' or 'non-uniform'.")

# 5. Compute parameters
CLUSTER_NAME = None
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job naming
RUN_LABEL = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and store the credentials in the current environment.


In [ ]:
from utils.auth import authenticate

await authenticate()


### 2.2. Initialize API Client


In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project


In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
PROJECT_ID = projects[0]["_id"]
print(f"Using project: {projects[0]['name']} ({PROJECT_ID})")


## 3. Load and save the material
### 3.1. Load material from local uploads or Standata


In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from utils.jupyterlite import load_material_from_folder
from utils.visualize import visualize_materials as visualize

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME)
)

visualize(material)


### 3.2. Save material to the platform


In [ ]:
from utils.api import get_or_create_material

saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)
print(f"Saved material ID: {saved_material.id}")


## 4. Build the convergence workflow
### 4.1. Load the total-energy workflow


In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from utils.visualize import visualize_workflow

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = f"{MY_WORKFLOW_NAME} {RUN_LABEL}"

visualize_workflow(workflow)


### 4.2. Add convergence directly in the notebook


In [ ]:
assert KGRID_CONVERGENCE_TYPE in {"uniform", "non-uniform"}

convergence_subworkflow = workflow.subworkflows[0]
convergence_subworkflow.add_convergence(
    parameter=CONVERGENCE_PARAMETER,
    parameter_initial=PARAMETER_INITIAL,
    parameter_increment=PARAMETER_INCREMENT,
    result="total_energy",
    result_initial=RESULT_INITIAL,
    condition=CONVERGENCE_CONDITION,
    operator="<",
    tolerance=TOLERANCE,
    max_occurrences=MAX_OCCURRENCES,
)

print(f"K-grid convergence type: {KGRID_CONVERGENCE_TYPE}")
print(f"Convergence parameter: {convergence_subworkflow.convergence_param}")
print(f"Convergence result: {convergence_subworkflow.convergence_result}")
visualize_workflow(workflow)


## 5. Create compute and job
### 5.1. Configure compute


In [ ]:
from mat3ra.ide.compute import Compute

clusters = client.clusters.list()
if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")


### 5.2. Create the job with the embedded convergence workflow


In [ ]:
from utils.api import create_job
from utils.generic import dict_to_namespace
from utils.visualize import display_JSON

job_name = f"{workflow.name} {saved_material.formula}"
job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=workflow,
    project_id=PROJECT_ID,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace(job_response)
job_id = job._id
print(f"Job created: {job_id}")
display_JSON(job_response)


## 6. Submit and wait


In [ ]:
client.jobs.submit(job_id)
print(f"Submitted job: {job_id}")


In [ ]:
from utils.api import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)


## 7. Read convergence results from `job.scopeTrack`
### 7.1. Fetch the finished job and inspect `scopeTrack`


In [ ]:
from utils.visualize import display_JSON

finished_job = client.jobs.get(job_id)
scope_track = finished_job.get("scopeTrack") or []
print(f"Job status: {finished_job.get('status')}")
print(f"scopeTrack entries: {len(scope_track)}")

if scope_track:
    display_JSON(scope_track)
else:
    print("scopeTrack is empty. If this is a finished convergence job, we likely need a dedicated property.")


### 7.2. Reconstruct the convergence series in the notebook


In [ ]:
def get_job_scope_track(job: dict) -> list[dict]:
    return job.get("scopeTrack") or []


def get_energy_convergence_series_from_job(job: dict, subworkflow_index: int = 0) -> list[dict]:
    scope_track = get_job_scope_track(job)
    if not scope_track:
        return []

    job_workflow = Workflow.create(job["workflow"])
    subworkflow = job_workflow.subworkflows[subworkflow_index]
    return subworkflow.convergence_series(scope_track)


def get_convergence_parameter_name_from_job(job: dict, subworkflow_index: int = 0) -> str | None:
    job_workflow = Workflow.create(job["workflow"])
    return job_workflow.subworkflows[subworkflow_index].convergence_param


def normalize_converged_kgrid(parameter_name: str | None, parameter_value):
    if parameter_value is None:
        return None
    if parameter_name == "N_k":
        return [parameter_value, parameter_value, parameter_value]
    if parameter_name == "N_k_nonuniform":
        return list(parameter_value)
    return parameter_value


def get_converged_kgrid_from_job(job: dict, subworkflow_index: int = 0):
    series = get_energy_convergence_series_from_job(job, subworkflow_index=subworkflow_index)
    if not series:
        return None
    parameter_name = get_convergence_parameter_name_from_job(job, subworkflow_index=subworkflow_index)
    parameter_value = series[-1]["param"]
    return normalize_converged_kgrid(parameter_name, parameter_value)


series = get_energy_convergence_series_from_job(finished_job)
convergence_parameter_name = get_convergence_parameter_name_from_job(finished_job)
converged_parameter_value = series[-1]["param"] if series else None
converged_kgrid = get_converged_kgrid_from_job(finished_job)

print("Convergence series:")
for item in series:
    print(item)

print(f"\nConvergence parameter: {convergence_parameter_name}")
print(f"Converged parameter value: {converged_parameter_value}")
print(f"Converged k-grid: {converged_kgrid}")


### 7.3. Result to reuse elsewhere
Use `converged_kgrid` in another notebook when setting the SCF k-grid for the production calculation.


In [ ]:
converged_kgrid
